# EECS 203 Study Group Matcher

Uses CP-SAT to automatically sort users into groups based on their preferences, then automatically compose draft emails to send to the groups.

In [ ]:
%pip install pandas ortools -q

In [ ]:
# Dependencies
import math
from collections import defaultdict

import pandas as pd
from ortools.sat.python import cp_model

In [ ]:
# Import data
data = pd.read_csv("student_data/F26 EECS 203 Study Group Form (Responses) - Form Responses 1.csv")

# Preview imported data
data.head()

In [ ]:
# Data formatting

opt_in = "Would you like to be matched with a study group? PLEASE change THIS response if you would like to opt out of our matching process BEFORE the night of the next deadline (9/11)."
style_col = "What style of study group would you prefer?"
meet_col = "How would you prefer to meet with your study group?"
near_col = "If you live on or near campus, would you prefer a study group with other students who live near you?"
live_col = "Where do you live?"
gender_pref_col = "Would you prefer a group that includes other students with your gender identity?"
gender_col = "What is your gender identity?"

group_size = 4

# Complete mapping for every survey response value currently in the form.
# .replace() leaves unmatched values alone (no NaNs).
normalize_dict = {
    # Opt-in
    "Yes, please match me with a group.": "Yes",
    
    # Study style
    "Highly efficient - getting our work done with minimal socializing.": "efficient",
    "A mix - we need to get our work done, but also have some time to chat and have fun together.": "mix",
    "Other": "other",

    # Meeting preference
    "In person": "in_person",
    "A mix of in person and online": "hybrid",
    "Online": "online",

    # Prefer students who live nearby
    "Yes": "yes",
    "No preference": "no_pref",
    "Not applicable": "na",

    # Where do you live
    "On or near Central Campus": "central",
    "On or near North Campus": "north",

    # Gender-identity group preference
    "Yes - the whole group if possible.": "whole_group",
    "It doesn't have to be the whole group, but I don't want to be the only one with my gender identity.": "not_alone",

    # Gender identity
    "Female": "female",
    "Male": "male",
    "Non-binary": "non_binary",
    "Prefer not to answer": "prefer_not",
}

# Normalize values in all columns
for col in data.columns:
    data[col] = data[col].replace(normalize_dict)

# Drop people who didn't opt in
data = data[data[opt_in] == "Yes"]

# Preview data
data.head()

In [ ]:
# Prepare groups for the CP-SAT model.

# 1) Index the people we need to place
people = data.reset_index(drop=True)  # renumber rows 0..n-1; drop=True discards the old index

# Get the sizes of all the groups.
n_people = len(people)  # how many people opted in
n_groups = math.ceil(n_people / group_size)  # number of groups needed (round up)
group_sizes = [group_size] * (n_people // group_size)  # e.g. 30 people, size 4 → seven 4s
print(group_sizes)

# If there are leftover people, add one smaller final group.
if n_people % group_size:  # leftover people that don't fill a full group
    group_sizes.append(n_people % group_size)  # add one smaller final group

print(f"{n_people} people → {n_groups} groups with sizes {group_sizes}")

In [ ]:
# Create a CP-SAT model.
model = cp_model.CpModel()  # empty CP-SAT model we'll add vars/constraints to
assign = {}  # dict of BoolVars keyed by (person_index, group_index)

# Create a decision variable for each person-group pair.
for i in range(n_people):  # for each person
    for g in range(n_groups):  # for each group
        assign[i, g] = model.NewBoolVar(f"person{i}_group{g}")  # 0/1: is person i in group g?

# For each person, assign exactly one group.
for i in range(n_people):
    model.AddExactlyOne(assign[i, g] for g in range(n_groups))  # each person in exactly one group

# For each group, assign the planned size. (Hard Constraint)
for g in range(n_groups):
    n_in_group = sum(assign[i, g] for i in range(n_people)) # Count the number of people in group g
    model.Add(n_in_group == group_sizes[g]) # Each group must have the planned size

In [ ]:
def minimize_unique(model, assign, values, n_groups, name, cost=1):
    """Soft preference: prefer groups where everyone shares the same value.

    For each group and each distinct value, create a 0/1 flag that is 1 iff
    that value appears in the group. Return those flags as penalty terms.
    Caller should Minimize(sum(penalties)).

    Args:
        model: The CP-SAT model we are building (constraints are added to this).
        assign: Dict of BoolVars, assign[i, g] = 1 if person i is in group g.
        values: Sequence of one attribute per person (e.g. people[style_col]),
            aligned with person indices 0 .. n_people-1.
        n_groups: Number of groups in the matching.
        name: Short label used in variable names (e.g. "style", "meet").
        cost: Cost of each unique value in a group.

    Returns:
        List of BoolVars to add to the objective. Each is 1 when a particular
        value appears in a particular group (so more unique values → higher cost).

    Example: all "mix" in a group → cost 1; mix+efficient → cost 2.
    """
    # Map each value -> list of person indices with that value
    ids_by_value = {}
    for i, value in enumerate(values):
        ids_by_value.setdefault(value, []).append(i)

    penalties = []
    for g in range(n_groups):
        for value, idxs in ids_by_value.items():
            # 0/1 flag: does this value appear in group g?
            value_in_group = model.NewBoolVar(f"{name}_g{g}_{value}")
            count = sum(assign[i, g] for i in idxs)

            # Flag must match reality: 1 iff count >= 1
            model.Add(count >= 1).OnlyEnforceIf(value_in_group)
            model.Add(count == 0).OnlyEnforceIf(value_in_group.Not())

            penalties.append(value_in_group * cost)
    return penalties

In [ ]:
penalties = []

# Style — prefer same study style within a group
penalties += minimize_unique(model, assign, people[style_col], n_groups, "style",cost=1)

# Meeting — same idea later, e.g.:
penalties += minimize_unique(model, assign, people[meet_col], n_groups, "meet",cost=20)

# Location
penalties += minimize_unique(model, assign, people[near_col], n_groups, "near",cost=10)

In [ ]:
# Gender — person-specific soft prefs (not the same as minimize_unique)
#   not_alone:    charge cost if this person is the only one of their gender in their group
#   whole_group:  charge cost for each differently-gendered person seated with them

# Define the cost when someone prefers not_alone but is alone in their gender.
not_alone_cost = 15

# Define the cost per differently-gendered teammate when someone prefers whole_group.
whole_group_cost = 5

# Make lists of genders and gender prefs (aligned with person indices).
genders = list(people[gender_col])
prefs = list(people[gender_pref_col])

# Map each gender -> person indices with that gender.
same_gender_ids = {}
for i, gender in enumerate(genders):
    # Skip people who preferred not to answer (no gender to match on).
    if gender == "prefer_not":
        continue

    # Add the person to the list of people with their gender.
    same_gender_ids.setdefault(gender, []).append(i)

# For each person with a stated gender, add soft penalties from their preference.
for i in range(n_people):
    gender = genders[i]
    pref = prefs[i]
    # Skip people who preferred not to answer (no gender to match on).
    if gender == "prefer_not":
        continue

    # Collect same-gender peers of person i.
    peers = [j for j in same_gender_ids[gender] if j != i]

    # not_alone: charge cost if i is in a group with no same-gender peer.
    if pref == "not_alone":
        for g in range(n_groups):
            # Flag: is at least one same-gender peer in group g?
            has_peer = model.NewBoolVar(f"gender_peer_p{i}_g{g}")
            peer_count = sum(assign[j, g] for j in peers)

            # If there is at least one same-gender peer in the group, the flag is true.
            model.Add(peer_count >= 1).OnlyEnforceIf(has_peer)

            # If there is no same-gender peer in the group, the flag is false.
            model.Add(peer_count == 0).OnlyEnforceIf(has_peer.Not())

            # Flag: is i alone in their gender in group g?
            # alone <=> (i in group g) AND (no same-gender peer in g)
            alone = model.NewBoolVar(f"gender_alone_p{i}_g{g}")

            # If i is in the group and there is no same-gender peer in the group, the flag is true.
            model.AddBoolAnd([assign[i, g], has_peer.Not()]).OnlyEnforceIf(alone)

            # If i is not in the group or there is a same-gender peer in the group, the flag is false.
            model.AddBoolOr([assign[i, g].Not(), has_peer]).OnlyEnforceIf(alone.Not())

            # Add the penalty if i is alone in their gender in group g.
            penalties.append(alone * not_alone_cost)

    # whole_group: charge cost for each differently-gendered person seated with i.
    if pref == "whole_group":
        # Collect people with a different stated gender than i.
        others = [
            j for j in range(n_people)
            if j != i and genders[j] != "prefer_not" and genders[j] != gender
        ]
        for g in range(n_groups):
            for j in others:
                # Flag: are i and j both in group g?
                # both <=> (i in g) AND (j in g)

                # Create a flag for whether i and j are both in group g.
                both = model.NewBoolVar(f"gender_coed_p{i}_p{j}_g{g}")

                # If i and j are both in group g, the flag is true.
                model.AddBoolAnd([assign[i, g], assign[j, g]]).OnlyEnforceIf(both)

                # If i or j is not in group g, the flag is false.
                model.AddBoolOr([assign[i, g].Not(), assign[j, g].Not()]).OnlyEnforceIf(both.Not())

                # Add the penalty if i and j are both in group g.
                penalties.append(both * whole_group_cost)

In [ ]:
# Minimize total soft costs (hard constraints still must hold)
model.Minimize(sum(penalties))

# Solve
solver = cp_model.CpSolver()
status = solver.Solve(model)
print(solver.StatusName(status), f"— objective: {int(solver.ObjectiveValue())}")

# Print the groups
for g in range(n_groups):
    members = [i for i in range(n_people) if solver.Value(assign[i, g])]
    styles_in_group = [people.loc[i, style_col] for i in members]
    locations_in_group = [people.loc[i, near_col] for i in members]
    print(f"\nGroup {g + 1} (size {len(members)}; styles={styles_in_group}; locations={locations_in_group})")
    for i in members:
        print(f"  id={people.loc[i, 'Email Address']:>3}  style={people.loc[i, style_col]} meet={people.loc[i, meet_col]} location={people.loc[i, near_col]}, gender={people.loc[i, gender_col]}, gender_pref={people.loc[i, gender_pref_col]}")

In [ ]:
# Automatically compose an email to each group; output to a file.

output_path = "study_group_emails.txt"
sections = []

for g in range(n_groups):
    members = [i for i in range(n_people) if solver.Value(assign[i, g])]
    emails = [people.loc[i, "Email Address"] for i in members]
    to_line = ", ".join(emails)
    roster = "\n".join(f"  - {email}" for email in emails)

    body = f"""Hi everyone,

You've been matched into an EECS 203 study group! Please introduce yourselves and find a time that works for your first meeting.

Your group members:
{roster}

Tips:
  - Reply-all to this email so everyone is on the thread.
  - Decide whether you'll meet in person, online, or a mix.
  - Share how you'd like to work together (problem sets, review, etc.).

If you have any questions, reply to this email or reach out to course staff.

Good luck this semester!
EECS 203 Course Staff
"""

    sections.append(
        f"{'=' * 60}\n"
        f"Group {g + 1}\n"
        f"{'=' * 60}\n"
        f"To: {to_line}\n"
        f"Subject: EECS 203 Study Group Match — Group {g + 1}\n\n"
        f"{body}"
    )

with open(output_path, "w") as f:
    f.write("\n".join(sections))

print(f"Wrote {n_groups} emails to {output_path}")
